# GeoZarr pyramid demo

Convert a slice of the OceanSODA-ETHZ-HR delta-fCO2 product (global, 0.25 degrees, Zarr v3) into a GeoZarr multiscale pyramid, validate it and look at two levels.

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

import geozarr_pyramid_maker as gpm

In [ ]:
gpm.configure_logging("INFO")

## Open the data

The full store has 1886 weekly steps. We take only the first 12 (about 47 MiB) so the demo runs in seconds. The source has no CRS information, so EPSG:4326 is inferred (with a warning) and the ascending latitude axis is flipped north-up (also with a warning).

In [ ]:
URL = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/ocean-soda/dfco2.zarr/"
ds = xr.open_zarr(URL).isel(time=slice(0, 12))
ds

In [ ]:
plan = ds.geozarr.plan()
print(plan)

In [ ]:
result = ds.geozarr.to_pyramid("oceansoda_dfco2.zarr", overwrite=True)
print(f"total: {result.timings['total']:.1f} s")

In [ ]:
print(result.validation)
gpm.is_valid(result.validation)

In [ ]:
tree = xr.open_datatree("oceansoda_dfco2.zarr", engine="zarr")
for name, node in tree.children.items():
    print(name, dict(node.ds.sizes), node.ds["dfco2"].encoding.get("shards"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), constrained_layout=True)
for ax, name in zip(axes, ("0", str(len(plan.levels) - 1)), strict=True):
    da = tree[name].ds["dfco2"].isel(time=0)
    x, y = da[da.dims[-1]].values, da[da.dims[-2]].values
    dx, dy = abs(x[1] - x[0]), abs(y[1] - y[0])
    extent = (x[0] - dx / 2, x[-1] + dx / 2, y[-1] - dy / 2, y[0] + dy / 2)
    im = ax.imshow(da.values, extent=extent, cmap="RdBu_r", vmin=-50, vmax=50)
    ax.set_title(f"level {name}: {da.shape[0]} x {da.shape[1]}")
fig.colorbar(im, ax=axes, label="dfco2 (uatm)", shrink=0.8)
plt.show()

## Preview in the browser

Run this in a terminal (the server blocks, so it is not a notebook cell):

```bash
geozarr-pyramid preview oceansoda_dfco2.zarr --serve
```